# Tutorial 3: Working with TENDL (TALYS-based Evaluated Nuclear Data Library)

## Overview

TENDL is a comprehensive nuclear data library that provides evaluated cross-sections for a wide range of isotopes. Unlike ENDF, which is manually evaluated, TENDL uses the TALYS nuclear reaction code to systematically generate evaluations.

### What you'll learn:
- What makes TENDL unique compared to other libraries
- How to download TENDL data from JANIS or directly from PSI
- How to load and visualize TENDL cross-sections
- How to compare TENDL with ENDF evaluations (real data!)
- Understanding the advantages and limitations of automated evaluations

### Prerequisites:
```bash
pip install matplotlib numpy pandas
```

### ⚠️ IMPORTANT: This tutorial requires REAL TENDL data!
You must download actual TENDL evaluations before running this notebook.

## 1. Understanding TENDL

### Key Features:

- **Automated Evaluation**: Uses TALYS code for consistent evaluations
- **Extensive Coverage**: Data for ~2800 isotopes (vs ~400 in ENDF)
- **Regular Updates**: Annual releases (TENDL-2021, TENDL-2023, etc.)
- **Format**: Uses ENDF-6 format (same as ENDF/B)
- **Use Cases**: Medical isotopes, fusion, activation studies

### TENDL vs ENDF:

| Feature | TENDL | ENDF |
|---------|-------|------|
| Evaluation Method | Automated (TALYS) | Manual expert evaluation |
| Isotope Coverage | ~2800 isotopes | ~400 isotopes |
| Quality for Major Isotopes | Good | Excellent |
| Uncertainties | Provided | Limited |
| Update Frequency | Annual | Every 3-5 years |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Create data directory
data_dir = Path('../data/tendl')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"Data directory: {data_dir.absolute()}")

## 2. 📥 DOWNLOADING REAL TENDL DATA

### Method 1: JANIS Web Interface (RECOMMENDED - Easiest!)

JANIS allows you to download TENDL data in CSV format:

1. **Visit JANIS:**
   - Go to: https://www.oecd-nea.org/janisweb/

2. **Search for Cross-Section Data:**
   - Click "Search" → "Cross Sections"
   - Select isotope: U-235
   - Select reaction: (n,f) for fission
   - **Important**: Check the "TENDL-2021" library checkbox
   - Click "Plot Data"

3. **Export Data:**
   - Click "Export" → "CSV format"
   - Save as: `u235_fission_tendl.csv`
   - Repeat for capture reaction (n,γ): Save as `u235_capture_tendl.csv`
   - Save both files to: `../data/tendl/`

### Method 2: Direct Download from PSI (Advanced)

For raw ENDF-6 files:

1. **Visit TENDL website:** https://tendl.web.psi.ch/
2. **Navigate to:** tendl_2021 → neutron_file
3. **Download:** n-U-235.tendl (or your isotope)
4. **Parse with tools:** Use `endf-parserpy` Python package

### 📋 Required Files (for this tutorial):
- `u235_fission_tendl.csv` - U-235 fission cross-section from TENDL-2021
- `u235_capture_tendl.csv` - U-235 capture cross-section from TENDL-2021
- Save both to: `../data/tendl/`

### 💡 Bonus: Medical Isotopes (Optional)
- Download Mo-99 (n,γ) cross-section from TENDL (medical isotope production)
- Save as: `mo99_ng_tendl.csv`

## 3. Loading Real TENDL Data

Let's load the TENDL data you downloaded from JANIS.

In [ ]:
# Define expected file paths
fission_file = data_dir / 'u235_fission_tendl.csv'
capture_file = data_dir / 'u235_capture_tendl.csv'

# Check for required files
missing_files = []
if not fission_file.exists():
    missing_files.append(fission_file.name)
if not capture_file.exists():
    missing_files.append(capture_file.name)

if missing_files:
    print("❌ ERROR: TENDL data files not found!")
    print(f"\nMissing files: {', '.join(missing_files)}")
    print(f"Expected directory: {data_dir.absolute()}")
    print("\n📥 PLEASE DOWNLOAD REAL TENDL DATA:")
    print("\n   Using JANIS (Recommended):")
    print("   1. Go to https://www.oecd-nea.org/janisweb/")
    print("   2. Click Search → Cross Sections")
    print("   3. Select U-235, reaction (n,f), library TENDL-2021")
    print("   4. Plot Data → Export → CSV")
    print("   5. Save as: u235_fission_tendl.csv")
    print("   6. Repeat for (n,γ) capture, save as: u235_capture_tendl.csv")
    print(f"   7. Save both to: {data_dir.absolute()}")
    print("\n   Alternative - Direct from PSI:")
    print("   1. Go to https://tendl.web.psi.ch/")
    print("   2. Navigate to tendl_2021/neutron_file/")
    print("   3. Download n-U-235.tendl")
    print("   4. Parse with endf-parserpy or export via JANIS")
    print("\n⚠️ This tutorial REQUIRES real TENDL data!")
    raise FileNotFoundError(f"Required TENDL files not found: {missing_files}")

print("✓ All required TENDL files found!")

## 4. Parse TENDL CSV Data from JANIS

Load and standardize the TENDL data exported from JANIS.

In [ ]:
def load_janis_csv(file_path, reaction_name):
    """
    Load TENDL data from JANIS CSV export.
    JANIS CSV format may vary, so we try to identify columns intelligently.
    """
    try:
        # Read CSV (JANIS may use different separators)
        df = pd.read_csv(file_path, comment='#', skipinitialspace=True)
        
        # Standardize column names (JANIS uses various formats)
        # Energy column
        energy_cols = [col for col in df.columns if 'energ' in col.lower() or 'e' in col.lower()]
        # Cross-section column  
        xs_cols = [col for col in df.columns if any(x in col.lower() for x in ['cross', 'xs', 'sigma', 'data'])]
        
        if not energy_cols or not xs_cols:
            print(f"⚠️ Warning: Could not identify columns in {file_path.name}")
            print(f"   Available columns: {df.columns.tolist()}")
            print(f"   Assuming first column is Energy, second is Cross-section")
            energy_col = df.columns[0]
            xs_col = df.columns[1]
        else:
            energy_col = energy_cols[0]
            xs_col = xs_cols[0]
        
        # Create standardized dataframe
        result = pd.DataFrame()
        result['Energy_eV'] = pd.to_numeric(df[energy_col], errors='coerce')
        result['CrossSection_barns'] = pd.to_numeric(df[xs_col], errors='coerce')
        
        # Remove NaN values
        result = result.dropna()
        result = result.sort_values('Energy_eV').reset_index(drop=True)
        
        print(f"✓ Loaded {reaction_name}: {len(result)} points [REAL DATA]")
        print(f"  Energy range: {result['Energy_eV'].min():.2e} - {result['Energy_eV'].max():.2e} eV")
        print(f"  XS range: {result['CrossSection_barns'].min():.3f} - {result['CrossSection_barns'].max():.3f} barns")
        
        return result
        
    except Exception as e:
        print(f"❌ Error loading {file_path.name}: {e}")
        raise

# Load TENDL fission data
print("Loading U-235 fission (TENDL-2021)...")
df_fission_tendl = load_janis_csv(fission_file, 'U-235 (n,f)')

print("\nLoading U-235 capture (TENDL-2021)...")
df_capture_tendl = load_janis_csv(capture_file, 'U-235 (n,γ)')

print("\n" + "="*60)
print("REAL TENDL DATA LOADED SUCCESSFULLY")
print("="*60)

## 5. Visualizing TENDL Cross-Sections [REAL DATA]

Let's plot the real TENDL evaluations for U-235.

In [ ]:
plt.figure(figsize=(14, 8))

plt.loglog(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'], 
           label='Fission (n,f) - MT=18', linewidth=2.5, color='red')
plt.loglog(df_capture_tendl['Energy_eV'], df_capture_tendl['CrossSection_barns'], 
           label='Capture (n,γ) - MT=102', linewidth=2.5, color='blue')

# Calculate absorption (fission + capture) if energy grids match
# Need to interpolate to common grid
from scipy.interpolate import interp1d
if len(df_fission_tendl) > 0 and len(df_capture_tendl) > 0:
    # Create common energy grid
    e_min = max(df_fission_tendl['Energy_eV'].min(), df_capture_tendl['Energy_eV'].min())
    e_max = min(df_fission_tendl['Energy_eV'].max(), df_capture_tendl['Energy_eV'].max())
    energy_common = np.logspace(np.log10(e_min), np.log10(e_max), 1000)
    
    # Interpolate both cross-sections
    f_fission = interp1d(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'], 
                         kind='linear', bounds_error=False, fill_value='extrapolate')
    f_capture = interp1d(df_capture_tendl['Energy_eV'], df_capture_tendl['CrossSection_barns'], 
                         kind='linear', bounds_error=False, fill_value='extrapolate')
    
    xs_fission_interp = f_fission(energy_common)
    xs_capture_interp = f_capture(energy_common)
    xs_absorption = xs_fission_interp + xs_capture_interp
    
    plt.loglog(energy_common, xs_absorption, 
               label='Absorption (Fission + Capture)', linewidth=2.5, 
               color='green', linestyle='--', alpha=0.7)

plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
plt.ylabel('Cross-section (barns)', fontsize=14, fontweight='bold')
plt.title('U-235 Neutron Cross-Sections (TENDL-2021) - REAL DATA', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, which='both')

# Add watermark
plt.text(0.98, 0.02, '[REAL DATA]', 
         transform=plt.gca().transAxes,
         fontsize=12, color='red', fontweight='bold',
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(data_dir / 'tendl_u235_cross_sections_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Plot saved: {data_dir / 'tendl_u235_cross_sections_REAL.png'}")

## 6. Comparing TENDL with ENDF [REAL DATA]

One of the most valuable uses of TENDL is to compare different evaluations. Let's compare real TENDL and ENDF data for U-235 fission.

In [ ]:
# Load ENDF data from Tutorial 1
endf_dir = Path('../data/endf')
endf_fission_file = endf_dir / 'u235_fission_endf8.csv'

if not endf_fission_file.exists():
    print("⚠️ WARNING: ENDF evaluation data not found!")
    print(f"   Expected file: {endf_fission_file}")
    print("\n   Please complete Tutorial 1 first to download ENDF data.")
    print("   For now, we'll skip the TENDL vs ENDF comparison.")
    df_fission_endf = None
else:
    try:
        df_fission_endf = pd.read_csv(endf_fission_file)
        print(f"✓ Loaded ENDF/B-VIII.0 evaluation: {len(df_fission_endf)} points [REAL DATA]")
    except Exception as e:
        print(f"❌ Error loading ENDF data: {e}")
        df_fission_endf = None

In [ ]:
# Plot comparison if ENDF data available
if df_fission_endf is not None:
    plt.figure(figsize=(14, 8))
    
    plt.loglog(df_fission_endf['Energy_eV'], df_fission_endf['CrossSection_barns'], 
               label='ENDF/B-VIII.0', linewidth=2.5, color='blue', alpha=0.8, zorder=1)
    plt.loglog(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'], 
               label='TENDL-2021', linewidth=2.5, color='red', linestyle='--', alpha=0.8, zorder=2)
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
    plt.ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
    plt.title('U-235 Fission Cross-Section: ENDF vs TENDL Comparison - REAL DATA', 
              fontsize=16, fontweight='bold', pad=20)
    plt.legend(fontsize=13)
    plt.grid(True, alpha=0.3, which='both')
    
    # Add watermark
    plt.text(0.98, 0.02, '[REAL DATA]', 
             transform=plt.gca().transAxes,
             fontsize=12, color='red', fontweight='bold',
             ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(data_dir / 'endf_vs_tendl_comparison_REAL.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Comparison saved: {data_dir / 'endf_vs_tendl_comparison_REAL.png'}")
else:
    print("⏭️ Skipping TENDL vs ENDF comparison (ENDF data not available)")

## 7. Ratio Plot: Quantifying Differences [REAL DATA]

A ratio plot helps visualize the relative differences between evaluations.

In [ ]:
if df_fission_endf is not None:
    # Interpolate TENDL to ENDF energy grid for direct comparison
    from scipy.interpolate import interp1d
    
    # Find common energy range
    e_min = max(df_fission_tendl['Energy_eV'].min(), df_fission_endf['Energy_eV'].min())
    e_max = min(df_fission_tendl['Energy_eV'].max(), df_fission_endf['Energy_eV'].max())
    
    # Filter ENDF data to common range
    mask = (df_fission_endf['Energy_eV'] >= e_min) & (df_fission_endf['Energy_eV'] <= e_max)
    df_endf_common = df_fission_endf[mask].copy()
    
    # Interpolate TENDL to ENDF energies
    f_tendl = interp1d(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'],
                       kind='linear', bounds_error=False, fill_value=np.nan)
    xs_tendl_interp = f_tendl(df_endf_common['Energy_eV'])
    
    # Calculate ratio
    ratio = xs_tendl_interp / df_endf_common['CrossSection_barns'].values
    
    # Create ratio plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), 
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    # Top panel: Cross-sections
    ax1.loglog(df_fission_endf['Energy_eV'], df_fission_endf['CrossSection_barns'], 
               label='ENDF/B-VIII.0', linewidth=2.5, color='blue', alpha=0.8)
    ax1.loglog(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'], 
               label='TENDL-2021', linewidth=2.5, color='red', linestyle='--', alpha=0.8)
    ax1.set_ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
    ax1.set_title('U-235 Fission: ENDF vs TENDL with Ratio - REAL DATA', 
                  fontsize=16, fontweight='bold', pad=20)
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3, which='both')
    
    # Bottom panel: Ratio
    ax2.semilogx(df_endf_common['Energy_eV'], ratio, linewidth=2, color='green')
    ax2.axhline(1.0, color='black', linestyle='-', linewidth=1.5, alpha=0.5)
    ax2.axhline(1.05, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='±5%')
    ax2.axhline(0.95, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Ratio\n(TENDL/ENDF)', fontsize=12, fontweight='bold')
    ax2.set_ylim(0.85, 1.15)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(data_dir / 'endf_tendl_ratio_REAL.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistics
    print("="*60)
    print("RATIO STATISTICS (TENDL/ENDF) [REAL DATA]")
    print("="*60)
    valid_ratio = ratio[~np.isnan(ratio)]
    print(f"Mean: {np.mean(valid_ratio):.4f}")
    print(f"Median: {np.median(valid_ratio):.4f}")
    print(f"Std Dev: {np.std(valid_ratio):.4f}")
    print(f"Min: {np.min(valid_ratio):.4f}")
    print(f"Max: {np.max(valid_ratio):.4f}")
    print(f"\nPoints within ±5%: {np.sum((valid_ratio >= 0.95) & (valid_ratio <= 1.05))} / {len(valid_ratio)} ({100*np.sum((valid_ratio >= 0.95) & (valid_ratio <= 1.05))/len(valid_ratio):.1f}%)")
    
    print(f"\n✓ Ratio plot saved: {data_dir / 'endf_tendl_ratio_REAL.png'}")
else:
    print("⏭️ Skipping ratio analysis (ENDF data not available)")

## 8. Medical Isotopes - TENDL's Strength [REAL DATA]

One of TENDL's strengths is coverage of isotopes not in ENDF, such as medical isotopes.

### Optional: Mo-99 Data

If you downloaded Mo-99 (n,γ) data from JANIS/TENDL, we can visualize it here.

In [ ]:
# Check for optional Mo-99 data
mo99_file = data_dir / 'mo99_ng_tendl.csv'

if mo99_file.exists():
    print("✓ Found Mo-99 data! Loading...")
    df_mo99 = load_janis_csv(mo99_file, 'Mo-99 (n,γ)')
    
    plt.figure(figsize=(14, 8))
    plt.loglog(df_mo99['Energy_eV'], df_mo99['CrossSection_barns'], 
               linewidth=2.5, color='purple')
    plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
    plt.ylabel('(n,γ) Cross-section (barns)', fontsize=14, fontweight='bold')
    plt.title('Mo-99(n,γ)Mo-100 Cross-Section from TENDL-2021 - REAL DATA\n(Medical Isotope Production)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.grid(True, alpha=0.3, which='both')
    
    # Add watermark
    plt.text(0.98, 0.02, '[REAL DATA]', 
             transform=plt.gca().transAxes,
             fontsize=12, color='red', fontweight='bold',
             ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(data_dir / 'mo99_tendl_REAL.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nMo-99 is important for:")
    print("  - Medical imaging (via Tc-99m decay product)")
    print("  - This data may only be available in TENDL, not ENDF")
    print(f"  - Thermal cross-section: {df_mo99['CrossSection_barns'].iloc[0]:.3f} barns")
else:
    print("ℹ️  Mo-99 data not found (optional).")
    print("   To explore medical isotopes:")
    print("   1. Go to JANIS: https://www.oecd-nea.org/janisweb/")
    print("   2. Search for Mo-99, (n,γ) reaction, TENDL-2021")
    print("   3. Export to CSV and save as mo99_ng_tendl.csv")

## 9. Creating ML Dataset [REAL DATA]

Let's create a comprehensive dataset combining TENDL and ENDF data for machine learning.

In [ ]:
# Create ML-ready dataset
if df_fission_endf is not None:
    # Interpolate both to common grid
    e_min = max(df_fission_tendl['Energy_eV'].min(), df_fission_endf['Energy_eV'].min())
    e_max = min(df_fission_tendl['Energy_eV'].max(), df_fission_endf['Energy_eV'].max())
    energy_ml = np.logspace(np.log10(e_min), np.log10(e_max), 5000)
    
    # Interpolate both datasets
    from scipy.interpolate import interp1d
    f_tendl_fis = interp1d(df_fission_tendl['Energy_eV'], df_fission_tendl['CrossSection_barns'], 
                           kind='linear', fill_value='extrapolate')
    f_endf_fis = interp1d(df_fission_endf['Energy_eV'], df_fission_endf['CrossSection_barns'], 
                          kind='linear', fill_value='extrapolate')
    f_tendl_cap = interp1d(df_capture_tendl['Energy_eV'], df_capture_tendl['CrossSection_barns'], 
                           kind='linear', fill_value='extrapolate')
    
    # Create comprehensive dataframe
    df_ml = pd.DataFrame({
        'Energy_eV': energy_ml,
        'TENDL_Fission_barns': f_tendl_fis(energy_ml),
        'TENDL_Capture_barns': f_tendl_cap(energy_ml),
        'ENDF_Fission_barns': f_endf_fis(energy_ml)
    })
    
    # Add derived features
    df_ml['TENDL_Absorption_barns'] = df_ml['TENDL_Fission_barns'] + df_ml['TENDL_Capture_barns']
    df_ml['Ratio_TENDL_ENDF'] = df_ml['TENDL_Fission_barns'] / df_ml['ENDF_Fission_barns']
    df_ml['Difference_barns'] = df_ml['TENDL_Fission_barns'] - df_ml['ENDF_Fission_barns']
    df_ml['Log10_Energy'] = np.log10(df_ml['Energy_eV'])
    
    # Classify energy regions
    def classify_energy_region(energy):
        if energy < 1:
            return 'Thermal'
        elif energy < 1e4:
            return 'Resonance'
        else:
            return 'Fast'
    
    df_ml['Energy_Region'] = df_ml['Energy_eV'].apply(classify_energy_region)
    
    # Save ML dataset
    ml_file = data_dir / 'tendl_endf_ML_dataset_REAL.csv'
    df_ml.to_csv(ml_file, index=False)
    
    print("="*60)
    print("ML DATASET PREPARED [REAL DATA]")
    print("="*60)
    print(f"\nTotal samples: {len(df_ml)}")
    print(f"\nFeatures included:")
    print(f"  - Energy (eV and log10)")
    print(f"  - TENDL fission, capture, absorption cross-sections")
    print(f"  - ENDF fission cross-section")
    print(f"  - Ratio (TENDL/ENDF)")
    print(f"  - Absolute difference")
    print(f"  - Energy region classification")
    print(f"\nData points by energy region:")
    print(df_ml['Energy_Region'].value_counts())
    print(f"\nRatio statistics by region:")
    print(df_ml.groupby('Energy_Region')['Ratio_TENDL_ENDF'].describe())
    print(f"\n✓ ML dataset saved: {ml_file}")
else:
    print("⏭️ Skipping ML dataset creation (ENDF data not available)")

## 10. Key Takeaways

### What We Learned:

1. **TENDL provides extensive coverage** - Many more isotopes than ENDF (~2800 vs ~400)
2. **Automated evaluation** - Consistent but may lack expert refinement for major isotopes
3. **Same format as ENDF** - Can use same tools for processing
4. **Includes uncertainties** - Valuable for ML uncertainty quantification
5. **Regularly updated** - New releases every 1-2 years
6. **Complementary to ENDF** - Use both for comprehensive analysis
7. **Excellent for rare isotopes** - Medical, activation, fusion applications

### When to Use TENDL:
- Working with exotic/rare isotopes not in ENDF
- Need systematic uncertainty data
- Medical isotope applications (Mo-99, I-131, etc.)
- Activation calculations
- Quick evaluations for new isotopes

### When to Use ENDF:
- Critical reactor calculations (safety-related)
- Major fissile isotopes (U-235, Pu-239, U-238)
- Need highest accuracy for well-studied isotopes
- Regulatory applications requiring certified data

### Important for Machine Learning:
- Use REAL data from both libraries for comprehensive training
- Compare evaluations to understand systematic uncertainties
- TENDL's extensive coverage helps with isotope interpolation/extrapolation
- Ratio analysis reveals where evaluations disagree

## Next Steps

- Download TENDL data for other isotopes (Pu-239, Fe-56, medical isotopes)
- Compare TENDL with other libraries (JEFF, JENDL) via Tutorial 4
- Use uncertainty data for Bayesian ML approaches
- Investigate specific energy regions where TENDL and ENDF differ
- Train ML models to predict nuclear data using multi-library comparisons

## Resources

- **TENDL Website:** https://tendl.web.psi.ch/
- **JANIS Interface:** https://www.oecd-nea.org/janisweb/
- **TALYS Code:** https://tendl.web.psi.ch/tendl_2021/talys.html
- **TENDL Documentation:** https://tendl.web.psi.ch/tendl_2021/tendl2021.html
- **endf-parserpy:** https://github.com/IAEA-NDS/endf-parserpy (for parsing raw files)